In [ ]:
from multiprocessing import freeze_support
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import pandas as pd

# ─────────────────────────────────────────
# 0. SEED
# ─────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    np.random.seed(42 + worker_id)
    random.seed(42 + worker_id)

set_seed(42)

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
INPUT_PATH  = "input_path"
OUTPUT_DIR  = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-4
IMG_SIZE    = (224, 224)
NUM_CLASSES = 3
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print(f"  Device      : {DEVICE}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Batch Size  : {BATCH_SIZE}")
print(f"  LR          : {LR}")
print(f"  Image Size  : {IMG_SIZE}")
print(f"  Num Classes : {NUM_CLASSES}")
print(f"  Output Dir  : {OUTPUT_DIR}")
print("=" * 60)

# ─────────────────────────────────────────
# 1. DATASET — stratified split
# ─────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(INPUT_PATH, transform=transform)
class_names  = full_dataset.classes
targets      = np.array(full_dataset.targets)
all_indices  = np.arange(len(full_dataset))

train_idx, temp_idx = train_test_split(all_indices, test_size=0.30, stratify=targets,         random_state=42)
val_idx,   test_idx = train_test_split(temp_idx,    test_size=0.50, stratify=targets[temp_idx], random_state=42)

train_set = Subset(full_dataset, train_idx)
val_set   = Subset(full_dataset, val_idx)
test_set  = Subset(full_dataset, test_idx)

print(f"\n📂 Dataset Split:")
print(f"   Total   : {len(full_dataset)}")
print(f"   Train   : {len(train_set)}")
print(f"   Val     : {len(val_set)}")
print(f"   Test    : {len(test_set)}")
print(f"   Classes : {class_names}")

g = torch.Generator(); g.manual_seed(42)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, worker_init_fn=seed_worker, generator=g)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

train_labels    = targets[train_idx]
class_weights   = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
class_weights_t = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print(f"\n⚖️  Class Weights:")
for i, (cls, w) in enumerate(zip(class_names, class_weights)):
    print(f"   [{i}] {cls:25s} → {w:.4f}")

# ─────────────────────────────────────────
# 2. SHARED BUILDING BLOCKS
# ─────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=(2, 3))
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

def unfreeze_last_n(model, n):
    for p in model.parameters():     p.requires_grad = False
    for p in list(model.parameters())[-n:]: p.requires_grad = True

# ─────────────────────────────────────────
# 3. ATTENTION MODULES
# ─────────────────────────────────────────
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_p = x.mean(dim=(2, 3))
        max_p = torch.amax(x, dim=(2, 3))
        ca    = torch.sigmoid(self.mlp(avg_p) + self.mlp(max_p))
        x     = x * ca.view(x.size(0), x.size(1), 1, 1)
        sa    = torch.cat([x.mean(dim=1, keepdim=True),
                           torch.amax(x, dim=1, keepdim=True)], dim=1)
        return x * self.sigmoid(self.conv(sa))


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )

    def forward(self, x):
        avg_p = x.mean(dim=(2, 3))
        max_p = torch.amax(x, dim=(2, 3))
        ca    = torch.sigmoid(self.mlp(avg_p) + self.mlp(max_p))
        return x * ca.view(x.size(0), x.size(1), 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        sa = torch.cat([x.mean(dim=1, keepdim=True),
                        torch.amax(x, dim=1, keepdim=True)], dim=1)
        return x * self.sigmoid(self.conv(sa))


class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.q     = nn.Linear(channels, channels)
        self.k     = nn.Linear(channels, channels)
        self.v     = nn.Linear(channels, channels)
        self.scale = channels ** -0.5

    def forward(self, x):
        b, c, h, w = x.size()
        f    = x.view(b, c, -1).transpose(1, 2)
        q, k, v = self.q(f), self.k(f), self.v(f)
        attn = torch.softmax(torch.bmm(q, k.transpose(1, 2)) * self.scale, dim=-1)
        out  = torch.bmm(attn, v).transpose(1, 2).view(b, c, h, w)
        return out + x


class MultiHeadAttention(nn.Module):
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.mha = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, x):
        b, c, h, w = x.size()
        f       = x.view(b, c, -1).transpose(1, 2)
        out, _  = self.mha(f, f, f)
        return out.transpose(1, 2).view(b, c, h, w) + x


class CrossAttention(nn.Module):
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.mha = nn.MultiheadAttention(channels, num_heads, batch_first=True)

    def forward(self, x):
        b, c, h, w = x.size()
        spatial = x.view(b, c, -1).transpose(1, 2)
        query   = x.mean(dim=(2, 3), keepdim=True).view(b, c, -1).transpose(1, 2)
        out, _  = self.mha(query, spatial, spatial)
        scale   = torch.sigmoid(out).transpose(1, 2).view(b, c, 1, 1)
        return x * scale


class CoordinateAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid         = max(8, channels // reduction)
        self.conv1  = nn.Conv2d(channels, mid, 1)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.ReLU()
        self.conv_h = nn.Conv2d(mid, channels, 1)
        self.conv_w = nn.Conv2d(mid, channels, 1)

    def forward(self, x):
        b, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y   = torch.cat([x_h, x_w], dim=2)
        y   = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = y.split([h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        return x * torch.sigmoid(self.conv_h(x_h)) * torch.sigmoid(self.conv_w(x_w))


class NonLocalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        mid        = channels // 2
        self.theta = nn.Conv2d(channels, mid, 1)
        self.phi   = nn.Conv2d(channels, mid, 1)
        self.g     = nn.Conv2d(channels, mid, 1)
        self.W     = nn.Conv2d(mid, channels, 1)
        self.bn    = nn.BatchNorm2d(channels)

    def forward(self, x):
        b, c, h, w = x.size()
        theta = self.theta(x).view(b, -1, h * w).permute(0, 2, 1)
        phi   = self.phi(x).view(b, -1, h * w)
        attn  = torch.softmax(torch.bmm(theta, phi), dim=-1)
        g     = self.g(x).view(b, -1, h * w).permute(0, 2, 1)
        y     = torch.bmm(attn, g).permute(0, 2, 1).view(b, c // 2, h, w)
        return x + self.bn(self.W(y))


class DualAttention(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.channel(x) + self.spatial(x)


# ─────────────────────────────────────────
# 4. FUSION MODEL
# ─────────────────────────────────────────
class FusionModel(nn.Module):
    def __init__(self, num_classes, attention_module):
        super().__init__()

        # ── DenseNet121 branch ──
        self.densenet = models.densenet121(pretrained=True)
        self.densenet.classifier = nn.Identity()
        unfreeze_last_n(self.densenet, 50)
        self.se_d   = SEBlock(1024)
        self.head_d = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )

        # ── EfficientNet-B0 branch (frozen) ──
        self.efficientnet = models.efficientnet_b0(pretrained=True)
        self.efficientnet.classifier = nn.Identity()
        for p in self.efficientnet.parameters():
            p.requires_grad = False
        self.se_e   = SEBlock(1280)
        self.head_e = nn.Sequential(
            nn.Linear(1280, 512), nn.BatchNorm1d(512), nn.Dropout(0.3), nn.ReLU(),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.Dropout(0.3), nn.ReLU()
        )

        # ── Fusion attention + classifier ──
        self.attention  = attention_module
        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        f_d = self.densenet(x).unsqueeze(-1).unsqueeze(-1)
        f_d = self.se_d(f_d).squeeze(-1).squeeze(-1)
        f_d = self.head_d(f_d)

        f_e = self.efficientnet(x).unsqueeze(-1).unsqueeze(-1)
        f_e = self.se_e(f_e).squeeze(-1).squeeze(-1)
        f_e = self.head_e(f_e)

        f = torch.cat([f_d, f_e], dim=1).unsqueeze(-1).unsqueeze(-1)
        f = self.attention(f).squeeze(-1).squeeze(-1)
        return self.classifier(f)


# ─────────────────────────────────────────
# 5. TRAIN / EVAL HELPERS
# ─────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    all_true, all_pred  = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y  = X.to(DEVICE), y.to(DEVICE)
            out   = model(X)
            loss  = criterion(out, y)
            preds = out.argmax(1)
            total_loss += loss.item() * X.size(0)
            correct    += (preds == y).sum().item()
            all_true.extend(y.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
    return total_loss / len(loader.dataset), correct / len(loader.dataset), all_true, all_pred


def print_separator(char="─", width=60):
    print(char * width)


# ─────────────────────────────────────────
# 6. RUN ALL VARIANTS
# ─────────────────────────────────────────
DIM = 512

attention_variants = {
    "CBAM"                 : CBAM(DIM),
    "Channel Attention"    : ChannelAttention(DIM),
    "Spatial Attention"    : SpatialAttention(),
    "Self-Attention"       : SelfAttention(DIM),
    "Multi-Head Attention" : MultiHeadAttention(DIM),
    "Cross-Attention"      : CrossAttention(DIM),
    "Coordinate Attention" : CoordinateAttention(DIM),
    "Non-Local Attention"  : NonLocalAttention(DIM),
    "Dual Attention"       : DualAttention(DIM),
}

results_summary   = []
classwise_summary = []
total_variants    = len(attention_variants)

for v_idx, (name, attn_module) in enumerate(attention_variants.items(), 1):

    print(f"\n")
    print_separator("═")
    print(f"  [{v_idx}/{total_variants}]  ATTENTION: {name}")
    print_separator("═")

    set_seed(42)
    model = FusionModel(NUM_CLASSES, attn_module).to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_p   = sum(p.numel() for p in model.parameters())
    print(f"  Params → Total: {total_p:,}  |  Trainable: {trainable:,}")

    criterion = nn.CrossEntropyLoss(weight=class_weights_t)
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

    best_val_acc           = 0.0
    best_state             = None
    train_accs, val_accs   = [], []
    train_losses, val_losses = [], []

    print_separator()
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc        = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _  = evaluate(model, val_loader, criterion)

        train_accs.append(tr_acc);    val_accs.append(vl_acc)
        train_losses.append(tr_loss); val_losses.append(vl_loss)

        best_flag = ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            best_flag    = "  ← best"

        print(f"  Epoch {epoch:02d}/{EPOCHS}  |  "
              f"Train  Acc: {tr_acc*100:6.2f}%  Loss: {tr_loss:.4f}  |  "
              f"Val    Acc: {vl_acc*100:6.2f}%  Loss: {vl_loss:.4f}{best_flag}")

    print_separator()

    # ── Test with best checkpoint ──
    model.load_state_dict(best_state)
    _, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)

    report_dict = classification_report(
        y_true, y_pred, target_names=class_names, digits=4, output_dict=True
    )
    report_str = classification_report(
        y_true, y_pred, target_names=class_names, digits=4
    )

    macro_p  = report_dict["macro avg"]["precision"]
    macro_r  = report_dict["macro avg"]["recall"]
    macro_f1 = report_dict["macro avg"]["f1-score"]

    # ── Confusion Matrix ──
    cm             = confusion_matrix(y_true, y_pred)
    total_samples  = len(y_true)

    # ── Per-class extended metrics ──
    per_class_metrics = []
    for i, cls in enumerate(class_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = total_samples - TP - FN - FP

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FNR = FN / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
        TNR = TN / (FP + TN) if (FP + TN) > 0 else 0

        cls_precision = report_dict[cls]["precision"]
        cls_f1        = report_dict[cls]["f1-score"]
        cls_acc       = (TP + TN) / total_samples
        cls_err       = 1 - cls_acc

        per_class_metrics.append({
            "Class"                    : cls,
            "Precision (%)"            : round(cls_precision * 100, 2),
            "Recall/TPR (%)"           : round(TPR * 100, 2),
            "F1-Score (%)"             : round(cls_f1 * 100, 2),
            "FNR (%)"                  : round(FNR * 100, 2),
            "FPR (%)"                  : round(FPR * 100, 2),
            "TNR (%)"                  : round(TNR * 100, 2),
            "Per Class Accuracy (%)"   : round(cls_acc * 100, 2),
            "Per Class Error Rate (%)" : round(cls_err * 100, 2),
        })

        classwise_summary.append({
            "Attention Mechanism"      : name,
            "Class"                    : cls,
            "Precision (%)"            : round(cls_precision * 100, 2),
            "Recall/TPR (%)"           : round(TPR * 100, 2),
            "F1-Score (%)"             : round(cls_f1 * 100, 2),
            "FNR (%)"                  : round(FNR * 100, 2),
            "FPR (%)"                  : round(FPR * 100, 2),
            "TNR (%)"                  : round(TNR * 100, 2),
            "Per Class Accuracy (%)"   : round(cls_acc * 100, 2),
            "Per Class Error Rate (%)" : round(cls_err * 100, 2),
        })

    # ── Print per-class table ──
    print(f"\n  📊 TEST RESULTS  [{name}]")
    print_separator()
    print(f"  Overall Accuracy : {test_acc*100:.2f}%")
    print(f"  Macro Precision  : {macro_p*100:.2f}%")
    print(f"  Macro Recall     : {macro_r*100:.2f}%")
    print(f"  Macro F1-Score   : {macro_f1*100:.2f}%")
    print_separator()

    col = [22, 12, 13, 11, 9, 9, 9, 18, 20]
    print(f"  {'Class':<{col[0]}}"
          f"{'Prec (%)':>{col[1]}}"
          f"{'Recall (%)':>{col[2]}}"
          f"{'F1 (%)':>{col[3]}}"
          f"{'FNR (%)':>{col[4]}}"
          f"{'FPR (%)':>{col[5]}}"
          f"{'TNR (%)':>{col[6]}}"
          f"{'Acc (%)':>{col[7]}}"
          f"{'Err Rate (%)':>{col[8]}}")
    print("  " + "─" * sum(col))

    for m in per_class_metrics:
        print(f"  {m['Class']:<{col[0]}}"
              f"{m['Precision (%)']:>{col[1]}.2f}"
              f"{m['Recall/TPR (%)']:>{col[2]}.2f}"
              f"{m['F1-Score (%)']:>{col[3]}.2f}"
              f"{m['FNR (%)']:>{col[4]}.2f}"
              f"{m['FPR (%)']:>{col[5]}.2f}"
              f"{m['TNR (%)']:>{col[6]}.2f}"
              f"{m['Per Class Accuracy (%)']:>{col[7]}.2f}"
              f"{m['Per Class Error Rate (%)']:>{col[8]}.2f}")

    print("  " + "─" * sum(col))
    print(f"\n  Full Classification Report:\n")
    for line in report_str.splitlines():
        print(f"    {line}")

    # ── Store overall results ──
    results_summary.append({
        "Attention Mechanism" : name,
        "Test Accuracy (%)"   : round(test_acc * 100, 2),
        "Macro Precision (%)" : round(macro_p  * 100, 2),
        "Macro Recall (%)"    : round(macro_r  * 100, 2),
        "Macro F1 (%)"        : round(macro_f1 * 100, 2),
    })

    safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "")

    # ── Save checkpoint ──
    torch.save(best_state, os.path.join(OUTPUT_DIR, f"best_{safe}.pth"))

    # ── Confusion matrix plot ──
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names, yticklabels=class_names, cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"cm_{safe}.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # ── Accuracy & Loss curves ──
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([a * 100 for a in train_accs], label="Train Acc")
    axes[0].plot([a * 100 for a in val_accs],   label="Val Acc")
    axes[0].set_title(f"Accuracy — {name}")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy (%)")
    axes[0].legend()
    axes[1].plot(train_losses, label="Train Loss")
    axes[1].plot(val_losses,   label="Val Loss")
    axes[1].set_title(f"Loss — {name}")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"curves_{safe}.png"), dpi=300, bbox_inches="tight")
    plt.close()

    del model
    torch.cuda.empty_cache()


# ─────────────────────────────────────────
# 7. FINAL SUMMARY — print + save CSV
# ─────────────────────────────────────────
df_overall   = pd.DataFrame(results_summary).sort_values("Macro F1 (%)", ascending=False).reset_index(drop=True)
df_classwise = pd.DataFrame(classwise_summary)

df_overall.to_csv(os.path.join(OUTPUT_DIR,   "summary_overall.csv"),   index=False)
df_classwise.to_csv(os.path.join(OUTPUT_DIR, "summary_classwise.csv"), index=False)

# ── Overall Summary Table ──
print(f"\n\n")
print("═" * 80)
print("  FINAL SUMMARY — OVERALL METRICS (sorted by Macro F1)")
print("═" * 80)
col_w = [26, 18, 20, 16, 14]
header = (f"  {'Attention Mechanism':<{col_w[0]}}"
          f"{'Test Acc (%)':>{col_w[1]}}"
          f"{'Macro Prec (%)':>{col_w[2]}}"
          f"{'Macro Rec (%)':>{col_w[3]}}"
          f"{'Macro F1 (%)':>{col_w[4]}}")
print(header)
print("  " + "─" * sum(col_w))
for _, row in df_overall.iterrows():
    print(f"  {row['Attention Mechanism']:<{col_w[0]}}"
          f"{row['Test Accuracy (%)']:>{col_w[1]}.2f}"
          f"{row['Macro Precision (%)']:>{col_w[2]}.2f}"
          f"{row['Macro Recall (%)']:>{col_w[3]}.2f}"
          f"{row['Macro F1 (%)']:>{col_w[4]}.2f}")
print("  " + "─" * sum(col_w))

# ── Class-wise Extended Summary Table ──
print(f"\n\n")
print("═" * 140)
print("  FINAL SUMMARY — CLASS-WISE EXTENDED METRICS")
print("═" * 140)

cw = [26, 22, 12, 13, 11, 9, 9, 9, 18, 20]
cheader = (f"  {'Attention Mechanism':<{cw[0]}}"
           f"{'Class':<{cw[1]}}"
           f"{'Prec (%)':>{cw[2]}}"
           f"{'Recall (%)':>{cw[3]}}"
           f"{'F1 (%)':>{cw[4]}}"
           f"{'FNR (%)':>{cw[5]}}"
           f"{'FPR (%)':>{cw[6]}}"
           f"{'TNR (%)':>{cw[7]}}"
           f"{'Acc (%)':>{cw[8]}}"
           f"{'Err Rate (%)':>{cw[9]}}")
print(cheader)
print("  " + "─" * sum(cw))

prev_attn = None
for _, row in df_classwise.iterrows():
    if prev_attn and prev_attn != row["Attention Mechanism"]:
        print("  " + "·" * sum(cw))
    prev_attn = row["Attention Mechanism"]
    print(f"  {row['Attention Mechanism']:<{cw[0]}}"
          f"{row['Class']:<{cw[1]}}"
          f"{row['Precision (%)']:>{cw[2]}.2f}"
          f"{row['Recall/TPR (%)']:>{cw[3]}.2f}"
          f"{row['F1-Score (%)']:>{cw[4]}.2f}"
          f"{row['FNR (%)']:>{cw[5]}.2f}"
          f"{row['FPR (%)']:>{cw[6]}.2f}"
          f"{row['TNR (%)']:>{cw[7]}.2f}"
          f"{row['Per Class Accuracy (%)']:>{cw[8]}.2f}"
          f"{row['Per Class Error Rate (%)']:>{cw[9]}.2f}")

print("  " + "─" * sum(cw))

# ── Comparison bar chart — Macro F1 ──
plt.figure(figsize=(13, 5))
bars = plt.bar(df_overall["Attention Mechanism"], df_overall["Macro F1 (%)"],
               color="steelblue", edgecolor="black")
plt.ylim(max(0, df_overall["Macro F1 (%)"].min() - 5), 100)
plt.ylabel("Macro F1 (%)"); plt.title("Attention Comparison — Macro F1 (sorted)")
plt.xticks(rotation=25, ha="right")
for bar, val in zip(bars, df_overall["Macro F1 (%)"]):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f"{val:.2f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_macro_f1.png"), dpi=300, bbox_inches="tight")
plt.close()

# ── Grouped bar — all 4 metrics ──
metrics = ["Test Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1 (%)"]
x = np.arange(len(df_overall)); w = 0.18
fig, ax = plt.subplots(figsize=(16, 6))
for i, m in enumerate(metrics):
    ax.bar(x + i * w, df_overall[m], width=w, label=m)
ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(df_overall["Attention Mechanism"], rotation=25, ha="right")
ax.set_ylabel("Score (%)"); ax.set_title("Attention Comparison — All Metrics")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comparison_all_metrics.png"), dpi=300, bbox_inches="tight")
plt.close()

print(f"\n✅ CSVs saved:")
print(f"   → {os.path.join(OUTPUT_DIR, 'summary_overall.csv')}")
print(f"   → {os.path.join(OUTPUT_DIR, 'summary_classwise.csv')}")
print(f"\n✅ All results saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    freeze_support()